# Credit Card Default Prediction

This notebook loads the dataset from the Kaggle source (`uciml/default-of-credit-card-clients-dataset`) and trains/tests a baseline model to predict whether a customer will default on the next payment cycle.

## 1) Load dataset

The notebook first tries Kaggle API download via `kagglehub`. If that is unavailable, place the dataset file locally (for example in `data/UCI_Credit_Card.csv` or `data/default of credit card clients.xls`) and rerun this cell.

In [ ]:
from pathlib import Path
import pandas as pd

DATASET_SLUG = "uciml/default-of-credit-card-clients-dataset"

def find_dataset_file(base_path: Path):
    candidates = list(base_path.rglob("*.csv")) + list(base_path.rglob("*.xls")) + list(base_path.rglob("*.xlsx"))
    if not candidates:
        raise FileNotFoundError("No CSV/XLS/XLSX file found in downloaded dataset path.")
    return candidates[0]

dataset_file = None
try:
    import kagglehub
    dataset_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
    dataset_file = find_dataset_file(dataset_dir)
    print(f"Loaded dataset directory from Kaggle: {dataset_dir}")
except Exception as exc:
    print(f"Kaggle download unavailable ({exc}). Falling back to local paths...")
    local_candidates = [
        Path("data/UCI_Credit_Card.csv"),
        Path("data/default of credit card clients.xls"),
        Path("UCI_Credit_Card.csv"),
        Path("default of credit card clients.xls"),
    ]
    for candidate in local_candidates:
        if candidate.exists():
            dataset_file = candidate
            break

if dataset_file is None:
    raise FileNotFoundError(
        "Dataset file not found. Download it from Kaggle and place it in ./data or install/configure kagglehub."
    )

if dataset_file.suffix.lower() == ".csv":
    df = pd.read_csv(dataset_file)
else:
    df = pd.read_excel(dataset_file, header=1)

print(f"Using dataset file: {dataset_file}")
df.head()

In [ ]:
df.shape, df.columns.tolist()[:10]

## 2) Prepare training data

In [ ]:
from sklearn.model_selection import train_test_split

target_col = "default.payment.next.month"
if target_col not in df.columns:
    raise KeyError(f"Expected target column `{target_col}` not found. Found: {df.columns.tolist()}")

X = df.drop(columns=[target_col])
if "ID" in X.columns:
    X = X.drop(columns=["ID"])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 3) Train and evaluate baseline model

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_score):.4f}")